# Thinking about contact surveys

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/16-contact-surveys.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

The next few chapters turn to empirical data on who contacts whom, and how to
feed it into a heterogeneous mixing model. It is easy to get this wrong by
applying assumptions that do not hold, so the first job is to be clear about
what the survey numbers actually represent.

## How contact surveys gather data

A contact survey asks participants to record, or recall, how many people with
particular characteristics they came into contact with over some period. The
seminal POLYMOD study
([Mossong *et al.*, *PLoS Medicine*, 2008](https://journals.plos.org/plosmedicine/article?id=10.1371/journal.pmed.0050074))
asked people in eight European countries to record their contacts over 24
hours, together with the characteristics of each person contacted and the
setting where they met. To keep things simple we look only at the *age* of the
people contacted, and ignore the setting.

## The nature of survey data

The first and most important point is that these numbers are fundamentally
different from the matrices of {doc}`14-assortative-mixing`, which assumed
density-dependent transmission. The survey asks how many people of a given kind
the *respondent* met, so the numbers are per capita only from the respondent's
side — not from the side of the person contacted.

To reason about this, split the population into just two groups, children and
adults (POLYMOD itself uses finer age bands). If a child reports meeting three
other children in a day, that is evidence that children typically meet about
three other children a day. It says nothing directly about how often two
*particular* children meet; that needs a further calculation.

So, by contrast with the earlier chapters, we are now working with
**frequency-dependent** contact rates. Otherwise the assumptions carry over:

- the matrix holds rates of social contact, and the infection flow's parameter
  is the risk of transmission per contact;
- the groups are mutually exclusive and together cover the whole modelled
  population;
- susceptibility and infectiousness are handled outside the matrix (or do not
  vary between groups);
- groups are ordered left to right and top to bottom, with **rows for the
  people being infected and columns for the people infecting them**. In
  summer4 that is {class}`~summer4.epi.ContactMatrix`'s orientation: row `i` is
  what participants in group `i` reported.

## Why the values are different

With these assumptions, the rate at which someone from one group (children)
meets people from another group (other children) depends — at least partly — on
how many people are in the other group. The rate at which two *specific*
people meet does not.

Take it to an extreme: a ship's passengers marooned on an island, 99 adults and
one child. Asked how many other children she met in the last 24 hours, the
child answers zero. The numbers a survey returns are shaped by the sizes of the
groups available to be contacted.

## Comparing to the mixing and transmission chapter

In {doc}`13-mixing-and-transmission-types` the population had size one and the
matrix held values below one, purely to show that density- and
frequency-dependent assumptions can coincide for some parameter choices. Those
values were not meant to be realistic.

Here we start from a frequency-dependent model that is still a simplification,
but with numbers closer to reality. POLYMOD found that respondents reported
about 13 contacts in 24 hours on average. Ignoring any empirical contact
structure for now, and taking one third of the population to be children,
homogeneous mixing gives every row the same split of those 13 contacts: a third
with children, two thirds with adults.

In [ ]:
import math

import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    FlowModel,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import ContactMatrix, ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

END_TIME = 40.0
POPULATION = 1.0
SEED = 0.01
RISK_PER_CONTACT = 0.05
INFECTIOUS_PERIOD = 4.0
PROP_CHILD = 1.0 / 3.0
TOTAL_CONTACTS = 13.0
CHILD_ADULT_BANDS = (0.0, 15.0, math.inf)  # children under 15, adults 15 and over

state = Property("state", ("susceptible", "infectious", "recovered"))
age = Property("age", ("child", "adult"))


def frequency_mixing(prop_child: float, total_contacts: float) -> ContactMatrix:
    """Homogeneous mixing: every row splits its contacts by population share."""
    row = [prop_child, 1.0 - prop_child]
    return ContactMatrix.from_array(total_contacts * np.array([row, row]), CHILD_ADULT_BANDS, prop=age)


homogeneous = frequency_mixing(PROP_CHILD, TOTAL_CONTACTS)
homogeneous.to_pandas()

Next, a familiar-looking SIR with frequency-dependent transmission.
`ForceOfInfection` defaults to `kind="frequency"`, so the force of infection on
group `i` is the risk per contact times $\sum_j c_{ij} I_j / N_j$.
`ContactMatrix.to_mixing` hands the matrix to it **without** row
normalisation: the row sums are the contact numbers, and normalising them
away would lose them.

The same function builds the stratified model (grouped by `age`) and an
unstratified one. An unstratified force of infection still needs a
one-trait grouping property and a $1 \times 1$ matrix, as in
{doc}`09-transmission-assumptions`.

In [ ]:
PLAN = SavePlan(
    requests={"infectious": SaveRequest(Compartments(where=state["infectious"]))},
    ts=np.linspace(0.0, END_TIME, 201),
)


def run_sir(group_by: Property, mixing: MixingMatrix, risk_per_contact: float) -> pd.Series:
    """Run a frequency-dependent SIR and return overall prevalence."""
    pmap = PropertyMap.from_property(state).stratify(group_by)
    model = FlowModel(pmap)
    infection = ForceOfInfection(
        "infection",
        infectious=state["infectious"],
        group_by=group_by,
        mixing=mixing,
        contact_rate=risk_per_contact,
    )
    model.add_flow(TransitionFlow("infection", state["susceptible"], state["infectious"], infection))
    model.add_flow(
        TransitionFlow("recovery", state["infectious"], state["recovered"], 1.0 / INFECTIOUS_PERIOD)
    )
    shares = [PROP_CHILD, 1.0 - PROP_CHILD] if group_by == age else [1.0]
    y0 = np.zeros(pmap.size)
    for trait, share in zip(group_by.traits, shares):
        y0[pmap.select(state["susceptible"] & group_by[trait])] = share * (POPULATION - SEED)
        y0[pmap.select(state["infectious"] & group_by[trait])] = share * SEED
    result = model.compile().run({}, y0, t0=0.0, t1=END_TIME, dt=0.05, save=PLAN, solver="euler")
    infectious = np.asarray(result["infectious"].values.data).sum(axis=1)
    return pd.Series(infectious / POPULATION, index=np.asarray(result["infectious"].times.values))

## Equivalent unstratified model

As in earlier chapters, check that an unstratified model reproduces the
stratified one. The unstratified model just needs the same number of daily
contacts: its risk per contact is multiplied by the 13 contacts a day.

In [ ]:
everyone = Property("everyone", ("all",))
outputs = pd.DataFrame(
    {
        "stratified": run_sir(age, homogeneous.to_mixing(), RISK_PER_CONTACT),
        "unstratified": run_sir(
            everyone,
            MixingMatrix(everyone, np.array([[1.0]]), normalize="none"),
            RISK_PER_CONTACT * TOTAL_CONTACTS,
        ),
    }
)
np.testing.assert_allclose(outputs["stratified"], outputs["unstratified"], rtol=1e-5)
outputs.plot(
    title="Stratified and unstratified models agree",
    labels={"index": "time", "value": "prevalence"},
)

The two models agree. In fact the stratified one would agree even if the left
column held some other proportion than the share of children: prevalence is the
same in both groups, so it does not matter how each row's contacts are divided
between them.

Notice already that this matrix is **not symmetric**, and under the
frequency-dependent assumption it should not be. It would be quite wrong to do
what we did at the end of {doc}`14-assortative-mixing` and average the
off-diagonal elements. Adults are twice as numerous as children here, so with
everyone mixing at random, children and adults alike would report meeting
twice as many adults as children — which is exactly what this matrix says.

## Implementing an empiric matrix

Now pretend we have survey data on how often people report meeting one
another. Keeping each row's total the same as before (13 contacts a day) keeps
the model equivalent to the previous ones.

In [ ]:
fake_survey = ContactMatrix.from_array([[9.0, 4.0], [2.0, 11.0]], CHILD_ADULT_BANDS, prop=age)
outputs["fake survey"] = run_sir(age, fake_survey.to_mixing(), RISK_PER_CONTACT)
np.testing.assert_allclose(outputs["fake survey"], outputs["stratified"], rtol=1e-5)
outputs.plot(
    title="A survey-like matrix with the same row totals",
    labels={"index": "time", "value": "prevalence"},
)

So we now have an "empiric, survey-based" matrix with some assortativity in it.
It does not change the overall dynamics yet, but it would as soon as the groups
differed epidemiologically in any other way.

One more thing about this matrix: the rate at which children report meeting
adults differs from the rate at which adults report meeting children — the
first is double the second. That is exactly as it should be, because there are
twice as many adults as children. The matrix is not symmetric in the
mathematical sense, but contacts are symmetric in the *social* sense: every time
a child reports meeting an adult, an adult reports meeting a child. Put
another way, the total contacts children report with adults equal the total
adults report with children, $c_{ij} N_i = c_{ji} N_j$.
{meth}`~summer4.epi.ContactMatrix.reciprocity_error` measures exactly that.

In [ ]:
shares = np.array([PROP_CHILD, 1.0 - PROP_CHILD])
reciprocity = fake_survey.reciprocity_error(shares)
assert reciprocity.relative < 1e-12
pd.DataFrame(
    {
        "children with adults (c_01 N_0)": [fake_survey.values[0, 1] * shares[0]],
        "adults with children (c_10 N_1)": [fake_survey.values[1, 0] * shares[1]],
        "relative reciprocity error": [reciprocity.relative],
    }
)

So the matrix is in a model, and we are ready to explore the dynamics of
heterogeneous mixing between children and adults.

But note that we should **not** take this matrix to a different epidemiological
setting, where children and adults are in different proportions, without
adapting it first. Suppose these estimates described Australia well, and we
believed social structures in Malaysia were similar but its population had a
different share of children. It would not be correct to change only the
modelled population structure and keep the same matrix.
{doc}`19-adapting-mixing-matrices` shows how to adapt it.